In [1]:
import time

from matplotlib.lines import Line2D
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import clear_output


In [13]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px


df = pd.read_csv(
    "ram_usage.csv",
    parse_dates=["timestamp"],
)

events = df.loc[df["event"].notna()]
ram = df.loc[df["event"].isna()]

fig = make_subplots(
    rows=1,
    cols=1,
    specs=[[{"secondary_y": True}]],
)

# % RAM used
fig.add_trace(
    go.Scatter(
        x=ram["timestamp"],
        y=ram["percent_used"],
        name="% RAM Used",
        line=dict(color="red"),
    ),
    secondary_y=False,
)

# Available RAM
fig.add_trace(
    go.Scatter(
        x=ram["timestamp"],
        y=ram["available_gb"],
        name="Available RAM (GB)",
        line=dict(color="blue"),
    ),
    secondary_y=True,
)

# Event markers
unique_events = sorted(events["event"].dropna().unique())

colours = px.colors.qualitative.Plotly
event_colours = {
    e: colours[i % len(colours)]
    for i, e in enumerate(unique_events)
}

for event in unique_events:

    subset = events.loc[events["event"] == event]

    first = True

    for _, row in subset.iterrows():

        fig.add_vline(
            x=row["timestamp"],
            line_dash="dash",
            line_color=event_colours[event],
            opacity=0.7,
        )

        fig.add_annotation(
            x=row["timestamp"],
            y=1,
            yref="paper",
            text=event,
            showarrow=False,
            textangle=-90,
            yanchor="top",
            xanchor="right",
            font=dict(
                size=10,
                color=event_colours[event],
            ),
        )

        # Dummy trace for legend entry
        if first:

            fig.add_trace(
                go.Scatter(
                    x=[None],
                    y=[None],
                    mode="lines",
                    line=dict(
                        color=event_colours[event],
                        dash="dash",
                    ),
                    name=event,
                )
            )

            first = False

fig.update_yaxes(
    title_text="% RAM Used",
    range=[0, 100],
    secondary_y=False,
)

fig.update_yaxes(
    title_text="Available RAM (GB)",
    secondary_y=True,
)

fig.update_layout(
    template="simple_white",
    width=1400,
    height=500,
    hovermode="x unified",
    legend=dict(
        orientation="h",
        y=1.2,
        x=0,
    ),
    margin=dict(t=100),
)

fig.show()


In [ ]:

while True:

    clear_output(wait=True)

    df = pd.read_csv(
        "ram_usage.csv",
        parse_dates=["timestamp"],
    )

    events = df.loc[df["event"].notna()]
    ram = df.loc[df["event"].isna()]

    fig, ax1 = plt.subplots(figsize=(18, 8))

    # % RAM used
    sns.lineplot(
        data=ram,
        x="timestamp",
        y="percent_used",
        ax=ax1,
        color="tab:red",
        label="% RAM Used",
    )

    ax1.set_ylabel("% RAM Used", color="tab:red")
    ax1.tick_params(axis="y", labelcolor="tab:red")
    ax1.set_ylim(0, 100)

    # Available RAM (GB)
    ax2 = ax1.twinx()

    sns.lineplot(
        data=ram,
        x="timestamp",
        y="available_gb",
        ax=ax2,
        color="tab:blue",
        label="Available RAM (GB)",
    )

    ax2.set_ylabel("Available RAM (GB)", color="tab:blue")
    ax2.tick_params(axis="y", labelcolor="tab:blue")

    unique_events = sorted(events["event"].dropna().unique())
    palette = sns.color_palette("tab10", n_colors=len(unique_events))

    event_colours = dict(zip(unique_events, palette))
    event_handles = []

    for event in unique_events:

        colour = event_colours[event]

        event_handles.append(
            Line2D(
                [0],
                [0],
                color=colour,
                linestyle="--",
                label=event,
            )
        )

        subset = events.loc[events["event"] == event]

        for _, row in subset.iterrows():

            ax1.axvline(
                row["timestamp"],
                color=colour,
                linestyle="--",
                alpha=0.7,
            )

            ax1.text(
                row["timestamp"],
                0.98,  # 98% up the axis
                event,
                rotation=90,
                va="top",
                ha="right",
                fontsize=8,
                color=colour,
                transform=ax1.get_xaxis_transform(),
            )

    # Keep only real legend entries
    line_handles = [
        h
        for h in (ax1.get_lines() + ax2.get_lines())
        if not h.get_label().startswith("_")
    ]

    handles = line_handles + event_handles

    ax1.legend(
        handles=handles,
        labels=[h.get_label() for h in handles],
        loc="upper left",
        ncol=2,
    )

    plt.tight_layout()
    plt.show()

    time.sleep(5)

In [ ]:
list(pd.read_parquet(r"C:\Users\Max.McFarlane\Code\MMI_European_Power_Modelling\data\France\processed\regional\France_FR10_ERA5_1_NUTS2_indexed.parquet").groupby(['latitude', 'longitude']))[0][1]